# Filtrando Grupos: HAVING e ANY

---

## Contexto: perguntas sobre totais e comparações

O gerente comercial chegou com novas perguntas e dessa vez elas envolvem **grupos** e **comparações com conjuntos**:

- *"Quais categorias têm mais de 3 produtos cadastrados?"*
- *"Quais clientes fizeram mais de 2 pedidos no total?"*
- *"Quais produtos têm preço maior que pelo menos um item vendido abaixo de R$50?"*

Para as duas primeiras, usamos **`HAVING`** que filtra *grupos* depois do `GROUP BY`.  
Para a terceira, usamos **`ANY`** que compara um valor com *qualquer elemento* de um conjunto.

Nesta aula você vai aprender:

- `HAVING` — filtrar resultados agrupados com condições sobre agregações
- `ANY` — comparar um valor com ao menos um elemento de uma subconsulta

---

## 1. Configuração

In [2]:
# Bibliotecas necessárias
from pathlib import Path
from datetime import datetime, timedelta
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect,
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index,
    union, union_all, exists, any_
)
from sqlalchemy.orm import (
    DeclarativeBase, Mapped, mapped_column, relationship, Session,
    joinedload, selectinload
)
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

Conexão OK ✅


In [3]:
# Verificar tabelas com SQLAlchemy
insp = inspect(engine)

print(insp.get_table_names())

['tb_clientes', 'tb_itens_pedidos', 'tb_pedidos', 'tb_produtos']


# 2. HAVING

O `HAVING` filtra grupos depois do `GROUP BY`, da mesma forma que o `WHERE` filtra linhas individuais, mas aplicado sobre o resultado de uma agregação.
**Por exemplo:** se você agrupar pedidos por cliente e contar quantos cada um fez, o `HAVING` permite manter só os grupos onde essa contagem é maior que 2. Isso não é possível com WHERE, porque a contagem ainda não existe quando o WHERE é aplicado.

Resumindo: `WHERE` filtra antes de agrupar, `HAVING` filtra depois.

In [9]:
# Quais categorias têm mais de 3 produtos cadastrados?

total_produtos = func.count(Produto.id) # conta quantos produtos existem

stmt = (
    select(
        Produto.categoria,
        total_produtos.label("total_produtos")
    )
    .group_by(Produto.categoria) # agrupa por categoria
    .having(total_produtos > 3) # filtra os grupos
    .order_by(total_produtos.desc())
)

# Debug
print(stmt.compile(compile_kwargs={"literal_binds": True}))
print("-" * 70)
with Session(engine) as session:
    resultados = session.execute(stmt).all()

    for row in resultados:
        print(f"{row.categoria}: {row.total_produtos} produtos")

SELECT tb_produtos.categoria, count(tb_produtos.id) AS total_produtos 
FROM tb_produtos GROUP BY tb_produtos.categoria 
HAVING count(tb_produtos.id) > 3 ORDER BY count(tb_produtos.id) DESC
----------------------------------------------------------------------
Roupas: 13 produtos
Esportes: 13 produtos
Livros: 11 produtos
Móveis: 7 produtos
Eletrônicos: 7 produtos


---

## 2. HAVING — filtrando grupos

### WHERE vs HAVING: qual é a diferença?

Essa é a confusão mais comum em SQL. A regra é simples:

| | `WHERE` | `HAVING` |
|---|---|---|
| Filtra | Linhas individuais | Grupos (resultado do GROUP BY) |
| Aplicado | **Antes** da agregação | **Depois** da agregação |
| Pode usar `func.count()`, `func.sum()`? | ❌ Não | ✅ Sim |

**Analogia:** imagine que você está contando votos por candidato.  
- `WHERE` descarta cédulas inválidas *antes* de contar.  
- `HAVING` elimina candidatos com poucos votos *depois* de contar.

### Caso de uso: categorias com mais de 3 produtos

## 2.1 Combinando WHERE e HAVING na mesma query

Os dois podem coexistir. `WHERE` filtra as linhas brutas; `HAVING` filtra os grupos resultantes.

**Caso de uso:** clientes ativos (com pedido não cancelado) que fizeram mais de 2 pedidos.

In [16]:
# WHERE filtra pedidos cancelados ANTES de agrupar.
# HAVING filtra os grupos com poucos pedidos DEPOIS de agrupar.

stmt = (
    select(
        Pedido.cliente_id,
        func.count(Pedido.id).label("total_pedidos"),
        func.sum(Pedido.valor_total).label("valor_acumulado")
    )
    .where(Pedido.status != "Cancelado")           # filtra LINHAS
    .group_by(Pedido.cliente_id)
    .having(func.count(Pedido.id) > 2)             # filtra GRUPOS
    .order_by(func.sum(Pedido.valor_total).desc())
)

print("SQL gerado:")
print(stmt)

with Session(engine) as session:
    grupos = session.execute(stmt).all()
    print(f"\nClientes com mais de 2 pedidos válidos: {len(grupos)}")
    for cliente_id, total, valor in grupos:
        print(f"  Cliente #{cliente_id} → {total} pedidos | R${valor:.2f} acumulado")

SQL gerado:
SELECT tb_pedidos.cliente_id, count(tb_pedidos.id) AS total_pedidos, sum(tb_pedidos.valor_total) AS valor_acumulado 
FROM tb_pedidos 
WHERE tb_pedidos.status != :status_1 GROUP BY tb_pedidos.cliente_id 
HAVING count(tb_pedidos.id) > :count_1 ORDER BY sum(tb_pedidos.valor_total) DESC

Clientes com mais de 2 pedidos válidos: 13
  Cliente #9 → 6 pedidos | R$316497.07 acumulado
  Cliente #4 → 7 pedidos | R$297870.90 acumulado
  Cliente #3 → 5 pedidos | R$291545.48 acumulado
  Cliente #21 → 5 pedidos | R$290720.27 acumulado
  Cliente #10 → 6 pedidos | R$273081.28 acumulado
  Cliente #6 → 5 pedidos | R$251422.56 acumulado
  Cliente #11 → 4 pedidos | R$233836.83 acumulado
  Cliente #5 → 6 pedidos | R$222184.27 acumulado
  Cliente #12 → 6 pedidos | R$219908.45 acumulado
  Cliente #20 → 3 pedidos | R$189873.59 acumulado
  Cliente #14 → 4 pedidos | R$181328.94 acumulado
  Cliente #2 → 3 pedidos | R$156045.52 acumulado
  Cliente #13 → 3 pedidos | R$53644.80 acumulado


## 2.2 Recuperando os objetos completos após o HAVING

O `HAVING` retorna apenas os campos do `GROUP BY`. Para buscar os objetos `Cliente` completos, basta usar `.in_()` com os IDs filtrados é um padrão que já vimos na aula anterior.

In [11]:
# Passo 1: subconsulta para obter apenas os IDs dos clientes qualificados
ids_clientes_freq = (
    select(Pedido.cliente_id)
    .where(Pedido.status != "Cancelado")
    .group_by(Pedido.cliente_id)
    .having(func.count(Pedido.id) > 2)
)

# Passo 2: buscar os objetos Cliente completos
stmt = select(Cliente).where(Cliente.id.in_(ids_clientes_freq))

with Session(engine) as session:
    clientes_freq = session.scalars(stmt).all()
    print(f"Clientes frequentes: {len(clientes_freq)}")
    for c in clientes_freq:
        print(f"  {c.nome} — {c.cidade}/{c.estado}")

Clientes frequentes: 13
  Srta. Julia Cassiano — São Paulo/SP
  Mathias Casa Grande — Salvador/BA
  Maria Sophia Guerra — Rio de Janeiro/RJ
  Stella Guerra — Brasília/DF
  Alícia Alves — Duque de Caxias/RJ
  Théo Cirino — Fortaleza/CE
  Luiz Miguel Fogaça — Goiânia/GO
  Dra. Evelyn Correia — Belo Horizonte/MG
  Sr. Mathias Guerra — São Gonçalo/RJ
  Yasmin Fonseca — São Gonçalo/RJ
  Joaquim Oliveira — São Luís/MA
  Davi Luiz Pimenta — Fortaleza/CE
  Ana Liz Monteiro — Duque de Caxias/RJ


**O padrão é sempre este:**
```
1. Montar a query com GROUP BY + HAVING   →  select(...).group_by(...).having(...)
2. Usar como subconsulta                  →  .in_(subconsulta)
3. Buscar os objetos completos            →  select(Modelo).where(...)
```

---

## 3. ANY — comparar com ao menos um valor do conjunto

### O que é ANY?

`ANY` responde: *"este valor é maior/menor/igual a **pelo menos um** valor dessa subconsulta?"*

Funciona com qualquer operador de comparação: `>`, `<`, `=`, `!=`, `>=`, `<=`.

| Expressão | Significado |
|---|---|
| `col > any_(subq)` | `col` é maior que **ao menos um** valor da subconsulta |
| `col = any_(subq)` | `col` é igual a **ao menos um** valor (equivale ao `IN`) |
| `col < any_(subq)` | `col` é menor que **ao menos um** valor |


> ⚠️ **Nota SQLite:** `ANY` é um operador do padrão SQL e funciona nativamente em PostgreSQL e MySQL. No SQLite, o SQLAlchemy emula o comportamento por baixo para produção com SQLite, prefira `IN` ou `EXISTS`.

### Caso de uso: produtos com preço acima de algum item vendido a preço baixo

In [12]:
# Subquery: valores dos pedidos cancelados
subq_cancelados = (
    select(Pedido.valor_total)
    .where(Pedido.status == "Cancelado")
)

# Query principal
stmt = (
    select(Cliente.nome, Pedido.valor_total)
    .join(Pedido)
    .where(Pedido.valor_total > any_(subq_cancelados))
)

print(stmt.compile(compile_kwargs={"literal_binds": True}))

with Session(engine) as session:
    resultados = session.execute(stmt).all()

    for nome, valor in resultados:
        print(f"{nome} fez um pedido de R$ {valor} acima de algum pedido cancelado")

/tmp/ipykernel_64340/2210763492.py:11: SAWarning: implicitly coercing SELECT object to scalar subquery; please use the .scalar_subquery() method to produce a scalar subquery.
  .where(Pedido.valor_total > any_(subq_cancelados))


SELECT tb_clientes.nome, tb_pedidos.valor_total 
FROM tb_clientes JOIN tb_pedidos ON tb_clientes.id = tb_pedidos.cliente_id 
WHERE tb_pedidos.valor_total > ANY (SELECT tb_pedidos.valor_total 
FROM tb_pedidos 
WHERE tb_pedidos.status = 'Cancelado')


OperationalError: (sqlite3.OperationalError) near "SELECT": syntax error
[SQL: SELECT tb_clientes.nome, tb_pedidos.valor_total 
FROM tb_clientes JOIN tb_pedidos ON tb_clientes.id = tb_pedidos.cliente_id 
WHERE tb_pedidos.valor_total > ANY (SELECT tb_pedidos.valor_total 
FROM tb_pedidos 
WHERE tb_pedidos.status = ?)]
[parameters: ('Cancelado',)]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [18]:
# alternativa ao ANY, vamos usar o MIN
# Subconsulta para obter o menor valor_total dos pedidos cancelados
subq = (
    select(func.min(Pedido.valor_total))  # Seleciona o valor mínimo de valor_total
    .where(Pedido.status == "Cancelado")  # Filtra apenas pedidos com status "Cancelado"
)

# Consulta principal: seleciona clientes cujos pedidos têm valor_total maior que o mínimo dos cancelados
stmt_min = (
    select(Cliente.nome, Pedido.valor_total)  # Seleciona o nome do cliente e o valor total do pedido
    .join(Pedido)  # Faz join com a tabela Pedido (assumindo relacionamento Cliente-Pedido)
    .where(Pedido.valor_total > subq.scalar_subquery())  # Filtra pedidos onde valor_total > mínimo dos cancelados
)

# Imprime o SQL gerado pela consulta (com valores literais para debug)
print(stmt_min.compile(compile_kwargs={"literal_binds": True}))
print("-" * 80)

# Executa a consulta no banco de dados
with Session(engine) as session:
    resultados = session.execute(stmt_min).all()  # Obtém todos os resultados como tuplas

    # Itera sobre os resultados e imprime cada cliente com seu pedido
    for nome, valor in resultados:
        print(f"{nome} fez um pedido de R$ {valor} acima de algum pedido cancelado")

SELECT tb_clientes.nome, tb_pedidos.valor_total 
FROM tb_clientes JOIN tb_pedidos ON tb_clientes.id = tb_pedidos.cliente_id 
WHERE tb_pedidos.valor_total > (SELECT min(tb_pedidos.valor_total) AS min_1 
FROM tb_pedidos 
WHERE tb_pedidos.status = 'Cancelado')
--------------------------------------------------------------------------------
Deborah Foroni fez um pedido de R$ 3499.90 acima de algum pedido cancelado
Maria Sophia Guerra fez um pedido de R$ 13120.80 acima de algum pedido cancelado
Laís Santos fez um pedido de R$ 110713.96 acima de algum pedido cancelado
Davi Luiz Pimenta fez um pedido de R$ 33159.28 acima de algum pedido cancelado
Théo Cirino fez um pedido de R$ 32498.76 acima de algum pedido cancelado
Dra. Evelyn Correia fez um pedido de R$ 65629.75 acima de algum pedido cancelado
Théo Cirino fez um pedido de R$ 78330.55 acima de algum pedido cancelado
Stella Guerra fez um pedido de R$ 19757.02 acima de algum pedido cancelado
Luiz Miguel Fogaça fez um pedido de R$ 65279.99 ac

---

## Exercício: Usando IA para isso

**Prompt para gerar um HAVING:**
```
Com os modelos SQLAlchemy abaixo, escreva uma query que retorne
os estados com mais de 2 clientes cadastrados, ordenados
pelo total de clientes decrescente.
Use GROUP BY + HAVING.

[modelos ORM]
```
---

### Resposta:Código gerado pelo Copilot

In [19]:
# Código para retornar estados com mais de 2 clientes cadastrados, ordenados pelo total decrescente
stmt = (
    select(
        Cliente.estado,  # Seleciona o estado
        func.count(Cliente.id).label("total_clientes")  # Conta o número de clientes por estado
    )
    .group_by(Cliente.estado)  # Agrupa por estado
    .having(func.count(Cliente.id) > 2)  # Filtra grupos com mais de 2 clientes
    .order_by(func.count(Cliente.id).desc())  # Ordena pelo total de clientes decrescente
)

# Executa a consulta
with Session(engine) as session:
    resultados = session.execute(stmt).all()
    for estado, total in resultados:
        print(f"{estado}: {total} clientes")

RJ: 5 clientes
SP: 4 clientes
